# Week 5 — Refresh / Content Opportunity Scoring: Decline Model (v3)

**Builds on:** Week 1–4 (research question, task framing, leakage-safe data contract, rule baseline).

**v3 change — `dim_content`:** most `content_created_date` values in this warehouse fall in **May–July
2026**, inside or after the `fact_content_query_90d` window (`2026-04-02` – `2026-06-30`). A fixed
Jan–Apr trailing window (v2) would silently mishandle most content: either zero trailing history, or
worse, a "decline" label on something that only existed for part of the label window. Eligibility is now
gated on `content_created_date`, and content age becomes a feature instead of an assumption.

**Also new:** `optimization_eligible_date` lets the final `action` recommendation respect FlyRank's own
scheduling constraint — no point recommending `urgent_review` today for something not eligible for
optimization yet; that becomes `monitor` instead. Small thing, but it's the difference between a model
output and an actual usable action engine.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

con = duckdb.connect()
# Accept the dataset gate in-browser first, then paste a READ token here.
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{BASE}/fact_content_daily_performance/**/*.parquet"
FACT_QUERY = f"{BASE}/fact_content_query_90d/**/*.parquet"
DIM_CLIENTS = f"{BASE}/dim_clients/**/*.parquet"
DIM_CONTENT = f"{BASE}/dim_content/**/*.parquet"


## Design

- **Prediction time** = end of `prev30`. `last30` is the label, everything else is a feature.
- **Eligibility now includes content age**: `content_created_date < WINDOW_START`, `is_published = true`,
  `is_deleted = false` — the item has to have actually existed before the analysis window to make a
  "declined" label meaningful.
- **Confound guard**: if `content_updated_date` (or `last_optimized_date`) falls *inside* the
  `prev30`/`last30` period, the item may have been manually refreshed mid-window — that's an
  intervention, not organic decay, and would confound an "organic decline" label. Flagged and excluded
  from training, kept in a separate check so you can see how many rows that actually affects.
- **New static features** (from `dim_content`, all knowable before the window since they're set at
  content-creation time): `search_volume`, `competition`, `cpc`, `backlinks`, `word_count`,
  `category_count`, `main_intent` (one-hot), `content_age_days`.
- **Operational gate on `action`**: `optimization_eligible_date` — a high-risk score doesn't become
  `urgent_review` unless FlyRank's own system says the content is actually eligible for optimization yet.


In [ ]:
WINDOW_START = "2026-04-02"
DAILY_FEATURE_START = "2025-01-02"   # generous trailing lookback; content_age_days will show how much of it each item actually had
DAILY_FEATURE_END = WINDOW_START

MIN_PREV30_IMPRESSIONS = 20
POSITION_WORSENING_THRESHOLD = 3.0
IMPRESSIONS_DROP_THRESHOLD = 0.20
CLICK_DROP_THRESHOLD = 0.20


### Eligible clients + eligible content (existed before the window, published, not deleted)

In [ ]:
eligible_clients_sql = f"""
    SELECT client_hash_id FROM read_parquet('{DIM_CLIENTS}')
    WHERE access_profile = 'gsc_and_ga4'
"""

eligible_content_sql = f"""
WITH eligible_clients AS ({eligible_clients_sql})
SELECT
    client_hash_id, content_hash_id,
    content_created_date, content_updated_date,
    content_type, main_intent,
    search_volume, competition, cpc, backlinks, word_count, category_count,
    last_optimized_date, optimization_eligible_date,
    DATE_DIFF('day', content_created_date, DATE '{WINDOW_START}') AS content_age_days,
    -- confound flag: updated/optimized during the analysis window itself
    (content_updated_date >= DATE '{WINDOW_START}') AS updated_during_window,
    (last_optimized_date IS NOT NULL AND last_optimized_date >= DATE '{WINDOW_START}') AS optimized_during_window
FROM read_parquet('{DIM_CONTENT}')
WHERE is_published = true AND is_deleted = false
  AND content_created_date < DATE '{WINDOW_START}'
  AND client_hash_id IN (SELECT client_hash_id FROM eligible_clients)
"""
eligible_content = con.sql(eligible_content_sql).df()
print(f"Eligible content items: {len(eligible_content)}")
print(f"Flagged as updated/optimized mid-window (excluded from training): "
      f"{(eligible_content['updated_during_window'] | eligible_content['optimized_during_window']).sum()}")


### Trailing daily features (strictly pre-`window_start`)

In [ ]:
daily_feat_sql = f"""
SELECT
    client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS impr_daily_trailing,
    SUM(gsc_clicks) AS clicks_daily_trailing,
    AVG(gsc_avg_position) AS avgpos_daily_trailing,
    SUM(ga4_engaged_sessions) AS engaged_daily_trailing,
    SUM(ga4_sessions) AS sessions_daily_trailing,
    SUM(sessions_ai) AS sessions_ai_daily_trailing,
    COUNT(*) AS days_observed_daily_trailing
FROM read_parquet('{FACT_DAILY}')
WHERE report_date >= '{DAILY_FEATURE_START}' AND report_date < '{DAILY_FEATURE_END}'
GROUP BY 1,2
"""
daily_feat = con.sql(daily_feat_sql).df()
print(daily_feat.shape)


### `prev30` features + `last30` vs `prev30` label (unchanged approach from v2)

In [ ]:
query_agg_sql = f"""
SELECT
    client_hash_id, content_hash_id,
    SUM(impressions_prev30) AS impr_prev30,
    SUM(clicks_prev30) AS clicks_prev30,
    SUM(avg_position_prev30 * impressions_prev30) / NULLIF(SUM(impressions_prev30), 0) AS avgpos_prev30_w,
    COUNT(DISTINCT CASE WHEN impressions_prev30 > 0 THEN query_hash_id END) AS visible_query_count_prev30,
    SUM(CASE WHEN impressions_prev30 > 0 AND impressions_prev30 < 10 THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(DISTINCT CASE WHEN impressions_prev30 > 0 THEN query_hash_id END), 0) AS rare_query_share_prev30,
    SUM(impressions_last30) AS impr_last30,
    SUM(clicks_last30) AS clicks_last30,
    SUM(avg_position_last30 * impressions_last30) / NULLIF(SUM(impressions_last30), 0) AS avgpos_last30_w
FROM read_parquet('{FACT_QUERY}')
GROUP BY 1,2
"""
query_agg = con.sql(query_agg_sql).df()
print(query_agg.shape)


### Assemble: eligible content ⋈ daily trailing ⋈ query prev30/last30, then build the label

In [ ]:
df = (
    eligible_content
    .merge(daily_feat, on=["client_hash_id", "content_hash_id"], how="left")
    .merge(query_agg, on=["client_hash_id", "content_hash_id"], how="inner")
)
df = df[df["impr_prev30"] >= MIN_PREV30_IMPRESSIONS].copy()

# exclude the mid-window-refresh confound from training data (organic-decline model only)
confounded = df["updated_during_window"] | df["optimized_during_window"]
df_clean = df[~confounded].copy()
print(f"Rows before confound exclusion: {len(df)} -> after: {len(df_clean)}")

impr_drop_frac = (df_clean["impr_prev30"] - df_clean["impr_last30"]) / df_clean["impr_prev30"].replace(0, np.nan)
click_drop_frac = (df_clean["clicks_prev30"] - df_clean["clicks_last30"]) / df_clean["clicks_prev30"].replace(0, np.nan)
position_worsened = (df_clean["avgpos_last30_w"] - df_clean["avgpos_prev30_w"]) >= POSITION_WORSENING_THRESHOLD
volume_dropped = (impr_drop_frac >= IMPRESSIONS_DROP_THRESHOLD) | (click_drop_frac >= CLICK_DROP_THRESHOLD)

df_clean["is_declining"] = (position_worsened & volume_dropped).astype(int)
print(df_clean["is_declining"].value_counts(normalize=True))


### Encode `main_intent`, finalize feature set

In [ ]:
df_clean = pd.get_dummies(df_clean, columns=["main_intent"], prefix="intent", dummy_na=True)
intent_cols = [c for c in df_clean.columns if c.startswith("intent_")]

safe_features = [
    "avgpos_daily_trailing", "impr_daily_trailing", "clicks_daily_trailing", "days_observed_daily_trailing",
    "avgpos_prev30_w", "impr_prev30", "clicks_prev30",
    "visible_query_count_prev30", "rare_query_share_prev30",
    "content_age_days", "search_volume", "competition", "cpc", "backlinks", "word_count", "category_count",
] + intent_cols

df_clean[["impr_daily_trailing", "clicks_daily_trailing", "days_observed_daily_trailing"]] = (
    df_clean[["impr_daily_trailing", "clicks_daily_trailing", "days_observed_daily_trailing"]].fillna(0)
)
work = df_clean.dropna(subset=[c for c in safe_features if not c.startswith("intent_")] + ["is_declining"]).reset_index(drop=True)
print(work.shape)


### Grouped train/test split (by `client_hash_id`)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(work, groups=work["client_hash_id"]))
train, test = work.iloc[train_idx].copy(), work.iloc[test_idx].copy()
assert set(train["client_hash_id"]) & set(test["client_hash_id"]) == set(), "Client leakage across split!"
print(f"Train: {len(train)} rows / {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test)} rows / {test['client_hash_id'].nunique()} clients")


### Baseline (recomputed on this exact test set) vs. model

In [ ]:
test["baseline_score"] = test["avgpos_prev30_w"] - test["avgpos_daily_trailing"]
baseline_top50 = test.sort_values("baseline_score", ascending=False).head(50)
baseline_precision_at_50 = baseline_top50["is_declining"].mean()

scaler = StandardScaler()
X_train = scaler.fit_transform(train[safe_features])
X_test = scaler.transform(test[safe_features])
model = LogisticRegression(class_weight="balanced", max_iter=1000).fit(X_train, train["is_declining"])
test["model_score"] = model.predict_proba(X_test)[:, 1]
model_top50 = test.sort_values("model_score", ascending=False).head(50)
model_precision_at_50 = model_top50["is_declining"].mean()

print(f"Baseline Precision@50: {baseline_precision_at_50:.3f}")
print(f"Model    Precision@50: {model_precision_at_50:.3f}")


### Reason codes + operational action gate

In [ ]:
coefs = pd.Series(model.coef_[0], index=safe_features)
contrib = pd.DataFrame(X_test, columns=safe_features, index=test.index) * coefs
test["reason_code"] = contrib.idxmax(axis=1)

test["optimization_eligible_now"] = (
    test["optimization_eligible_date"].notna() & (test["optimization_eligible_date"] <= pd.Timestamp(WINDOW_START))
)

def action_for(row):
    if not row["optimization_eligible_now"]:
        return "monitor"   # can't act on it yet regardless of score -- respects FlyRank's own scheduling
    if row["model_score"] >= 0.7:
        return "urgent_review"
    if row["model_score"] >= 0.4:
        return "scheduled_refresh"
    return "monitor"

test["action"] = test.apply(action_for, axis=1)


### Write ranked output (repo deliverable)

In [ ]:
out = (
    test.sort_values("model_score", ascending=False)
    .loc[:, ["client_hash_id", "content_hash_id", "model_score", "is_declining", "reason_code",
             "optimization_eligible_now", "action"]]
    .reset_index(drop=True)
)
out.insert(0, "rank", out.index + 1)
out.to_csv("../outputs/model_action_score.csv", index=False)
out.head(20)


## Limitations & honest framing (carry into the paper)

- **Directional, not causal** — association between pre-label signal and a `last30` outcome, not a claim
  about *why*, and not a claim about reverse-engineering Google's algorithm.
- **Confound exclusion is a judgment call, not a guarantee** — excluding mid-window-updated content
  removes the most obvious intervention confound, but doesn't rule out other unobserved interventions
  (e.g. a client-side redesign not captured in this warehouse).
- **Click sparsity at query grain** — the label leans on `impressions` drop as primary volume signal,
  clicks as secondary; report the actual `is_declining` positive rate before trusting Precision@50.
- **New/young content is systematically excluded** by the `content_created_date < window_start` filter —
  state clearly that this model is scoped to *existing* content, not brand-new AI-generated pages that
  haven't accumulated any trailing signal yet.
- **Threshold judgment calls** — `POSITION_WORSENING_THRESHOLD`, `IMPRESSIONS_DROP_THRESHOLD`,
  `CLICK_DROP_THRESHOLD` — state them explicitly; show the model still beats baseline under at least one
  alternate setting as a robustness check.
- **`optimization_eligible_now` gate is a real operational constraint**, not a modeling choice — say so,
  since it changes the action distribution independent of the model's actual predictive quality.
